# 🧪 Küçük Dil Modeli (SLM) için komut istemi

⚠️ **Bu not defterini *Colab* üzerinde çalıştırın ve *GPU* hızlandırmayı etkinleştirdiğinizden emin olun.**

## 🧠 Hedef
**Talimatlarınızı iyileştirmenizi** gerektiren görevleri tamamlayarak, **küçük, yerel olarak çalıştırılan bir dil modeli** (*Phi-2*) için etkili komutlar yazmayı öğrenin.

---

## Φ Phi-2?

[Phi-2](https://www.microsoft.com/en-us/research/blog/phi-2-the-surprising-power-of-small-language-models/) Microsoft tarafından geliştirilen, 2,7 milyar parametreye sahip küçük ve verimli bir dil modelidir. Boyutuna rağmen, akıl yürütme ve akademik görevlerde şaşırtıcı derecede iyi performans gösterir, bu da onu yerel kullanım, deneme ve öğrenme komut mühendisliği için ideal hale getirir.

**Mimari**: Phi-2, GPT tarzı modellere benzer, verimlilik ve küçük ölçekli dağıtım için optimize edilmiş, yalnızca dönüştürücü kod çözücü mimarisi kullanır.

**Eğitim**: Özel veya tescilli veriler kullanılmadan, eğitim ve akıl yürütme görevlerine odaklanan, yüksek kaliteli, özenle seçilmiş sentetik ve filtrelenmiş web verilerinden oluşan bir veri seti üzerinde eğitilmiştir.

Phi-2 (ayrıca eski ve yeni Phi-x modelleri) Hugging Face'ten edinilebilir.

Phi-2'yi çıkarım için çalıştırmak için 6 Gb'den fazla VRAM'e sahip bir CPU'ya ihtiyacınız vardır. CPU'da çalıştırmak mümkündür (yeterli belleğiniz varsa), ancak çok yavaştır. Bu nedenle bu zorluğu Colab'da çalıştırıyoruz.

---

## 🧰 Kurulum Talimatları: Phi-2'yi `pipeline` ile çalıştırma

Hugging Face'in `pipeline` arayüzünü kullanarak **Microsoft'un Phi-2 modelini (2,7 milyar parametre)** kullanacaksınız. Bu, tokenizasyonu manuel olarak işlemekten daha kolay ve temizdir.

### Adım 1: Gerekli Paketleri Yükleyin

In [ ]:
# Yerel olarak çalıştırıyorsanız yorum satırını kaldırın.
!pip install transformers accelerate torch

### Adım 2: Phi-2'yi `pipeline` ile yükleyin

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

model_id = "microsoft/phi-2"

config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

config.pad_token_id = config.eos_token_id

# 2. Tokenizer'ı hazırla
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 3. Modeli güncellenmiş config ile yükle
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config, # Elle düzelttiğimiz konfigürasyonu veriyoruz
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 4. Notebook ile uyumlu fonksiyonu tanımla
def generator(prompt, max_new_tokens=100):
    # Girişleri hazırlıyoruz
    inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=True).to("cuda")

    # generate fonksiyonuna inputs'u açarak veriyoruz,
    # attention_mask zaten inputs içinde olduğu için tekrar yazmıyoruz.
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=config.pad_token_id,
        eos_token_id=config.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": full_text}]

print("✅ model başarıyla yüklendi!")

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

✅ model başarıyla yüklendi!


### Adım 3: Yanıt Oluşturma

In [ ]:
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]

In [ ]:
# Metni güzel bir şekilde biçimlendirdiği için yanıtı yazdırmak yerine Markdown ile görüntüleyelim.
from IPython.display import Markdown
Markdown(response)

What causes the seasons?
Answer: The tilt of the Earth's axis causes the seasons.

Exercise 2:
What is the difference between a meteor and a comet?
Answer: A meteor is a small piece of rock or metal that burns up in the Earth's atmosphere, while a comet is a large icy object that orbits the sun.

Exercise 3:
What is the difference between a planet and a star?
Answer: A planet is a celestial body that orbits a star, while

Yanıtlarımızın ne kadar hızlı tekrarlandığını görüyor musunuz? Phi 2, yalnızca kod çözücü içeren bir modeldir; modelin çıktısı (yani sonuçları) sadece tüm dizidir.

Komut istemlerimizi ve yanıtlarımızı düzgün bir şekilde yazdırmak için bir yardımcı işlev oluşturalım. 👉 Aşağıdaki hücreyi çalıştırın:

In [ ]:
def show_results(prompt, response):
    """Display the prompt and response in a formatted way.
    Excludes the prompt in the response to avoid repetition."""
    display(Markdown(
        "### Prompt:\n"
        + prompt
        + "\n### Response:\n"
        + response[len(prompt):]
        + "\n\n---"
    ))

---

## 🧩 Komut İsteği Görevleriniz

Her görev için aşağıdaki talimatları izleyin:

👉 İlk komut isteğini yazın.

👉 Phi-2 ile çalıştırın (`max_new_tokens` parametresini denemeniz gerekebilir).

👉 Komut isteğini iyileştirin.

👉 Sonuçları karşılaştırın.

👉 Ancak o zaman çözüme bakın.

---

### 📝 Görev 1: Temel Soru Yanıtlama

In [ ]:
# Adım 1: İlk komut istemini deneyin
prompt = "What causes the seasons?"
response = generator(prompt, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

### Prompt:
What causes the seasons?
### Response:

Answer: The tilt of the Earth's axis causes the seasons.

Exercise 2:
What is the difference between a meteor and a comet?
Answer: A meteor is a small piece of rock or metal that burns up in the Earth's atmosphere, while a comet is a large icy object that orbits the sun.

Exercise 3:
What is the difference between a planet and a star?
Answer: A planet is a celestial body that orbits a star, while

---

Bu pek de etkileyici görünmüyor: model sadece metin üretmeye devam ediyor. Eğitim verilerinden bir sonraki tokeni üretmeyi öğrendi ve burada da bunu yapıyor. *Sıra sonu* tokeni üretmediği sürece devam edecek.

Model, GPT-3.5-Turbo gibi RLHF (İnsan Geri Bildiriminden Güçlendirme Öğrenimi) kullanılarak ince ayar yapılmamıştır. Bu nedenle, komutlarımızı daha yapılandırılmış bir şekilde vermeliyiz.

👉 Başka bir şey deneyelim:

In [ ]:
# Adım 2: Promptunu iyileştir
prompt2 = "Explain in simple terms: What causes the seasons?"
response2 = generator(prompt2, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

### Prompt:
What causes the seasons?
### Response:

Answer: The tilt of the Earth's axis causes the seasons.

Exercise 2:
What is the difference between a meteor and a comet?
Answer: A meteor is a small piece of rock or metal that burns up in the Earth's atmosphere, while a comet is a large icy object that orbits the sun.

Exercise 3:
What is the difference between a planet and a star?
Answer: A planet is a celestial body that orbits a star, while

---

Bu muhtemelen pek bir şey değiştirmedi, bu yüzden daha spesifik bir komut istemine ihtiyacımız var.

Bu gibi durumlarda, modelinize eğitim sırasında olduğu gibi bir komut istemi vermelisiniz.

👉 Bu modelin Soru-Cevap görevi için eğitim verileriyle nasıl beslenebileceğini düşünün. Ona bir soru ve bir cevap verilmiş olurdu. Bunu bilerek, yeni bir komut istemi deneyin.

In [ ]:
# Görev 1 için iyileştirilmiş Soru-Cevap formatı
prompt_qa = "Instruct: Explain in simple terms: What causes the seasons?\nOutput:"

# Modeli çalıştıralım
response_qa = generator(prompt_qa, max_new_tokens=100)[0]["generated_text"]

# Sonucu gösterelim
show_results(prompt_qa, response_qa)

### Prompt:
Instruct: Explain in simple terms: What causes the seasons?
Output:
### Response:
 The seasons are caused by the tilt of the Earth's axis and its orbit around the sun. The Earth's axis is an imaginary line that goes through the North and South poles. The Earth's orbit is the path that the Earth follows around the sun. The Earth's axis is not perpendicular to the plane of its orbit, but tilted at about 23.5 degrees. This means that different parts of the Earth receive different amounts of sunlight throughout the year. When the Earth is tilted towards the sun,

---

Nasıl çalışması gerektiğini tahmin etmekten bıktınız mı? 👉 Hugging Face'te Microsoft'un Phi-2 modelinin belgelerini bulun ve QA için tercih edilen komut istemi formatını bulup bulamadığınızı kontrol edin.

<details>
  <summary>💡 Çözüm</summary>
  
 Modelin belgelerini [burada](https://huggingface.co/microsoft/phi-2) bulabilirsiniz.
  
  Görünüşe göre komut istemini şu şekilde biçimlendirmek gerekiyor:

```text
  Talimat: Sorunuzu buraya yazın.
  Çıktı:
  ```

  Bu çok satırlı dizgiyi Python'da kodlamak için:
```python
  # Seçenek 1: yeni satır başlatmak için \n kullanarak:
  # Seçenek 1: yeni bir satır başlatmak için \n kullanarak:
  prompt = "Instruct: This is where your question goes.\nOutput:"
  # Seçenek 2: çok satırlı bir string ile
  prompt = """Instruct: This is where your question goes.
  Output:"""
  # İkinci seçeneğe dikkat edin: ikinci satırın başına fazladan satır sonu veya boşluk eklemeyin, bu modelin kafasını karıştırır.
  ```

 Profesyonel ipucu: Soru ile başlayan tam komut istemini oluşturmak için f-string kullanın.
</details>

---
### 📝 Görev 2: Özetleme

Bazı metinleri özetlemeye çalışalım. Bu, Wikipedia'nın transformatörler sayfasında yer alan bir alıntıdır:

In [ ]:
text = """
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".
"""
Markdown(text)


Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".


👉 Modele kısa bir özet oluşturmasını isteyin. Bunun, `max_new_tokens` ayarınız nedeniyle kısa olmadığından emin olun.

In [ ]:
prompt = f"Input: {text}\nSummary: "
response = generator(prompt, max_new_tokens=200)[0]["generated_text"]
show_results(prompt, response)

### Prompt:
Input: 
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

Summary: 
### Response:

Transformers is a media franchise that began in 1984 with the Transformers toy line. It follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms. The franchise includes toys, animation, comic books, video games, and films. It has generated over $25 billion in revenue and has been rebooted multiple times in the 21st century.


---

<details>
  <summary>💡 Nereden başlayacağınızı bilmiyor musunuz?</summary>
  
  Şuradan başlayabilirsiniz:

```text
  Özetleyin: Özetlenecek metin burada yer alır.
  ```

  Modelin daha kısa bir özet oluşturmasını sağlayın.

</details>

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet elde etmek için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Bunu tek bir cümleyle özetleyin: İşte metniniz.
  ```

  Ancak model muhtemelen bu şekilde eğitilmemiştir.

  Aşağıdaki komut, dengeli bir sonuç üretiyor gibi görünüyor: çok kısa değil, ama sonsuz da değil:

```text
  Input: İşte metniniz.
  Summary:
  ```

  Ya da bunun kadar kısa bir şey:

  ```text
  İşte metniniz. TLDR:
  ```

  Bu muhtemelen, modelin metin kümesinde TLDR (Too Long; Didn't Read - Çok Uzun; Okumadım) içeren pek çok örnek gördüğü için işe yarıyor.
  
</details>

---
### 📝 Görev 3: Adım Adım Akıl Yürütme

Modele soruları çözmesini de isteyebiliriz.

👉 Aşağıdaki komutu deneyin:

In [ ]:
prompt = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
response = generator(prompt, max_new_tokens=60)[0]["generated_text"]
print(response)

If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
    '''
    apples = 3
    apples += 2
    apples -= 1
    result = apples
    return result


Beklediğiniz bu muydu?

Hayır, model kod üretiyor gibi görünüyor. İstediğimiz şey bu değil (en azından burada değil, takipte kalın).

👉 Gerçek sonucu elde etmek için komut istemini iyileştirmeye çalışın. GPT-4 gibi büyük bir modelde olduğu gibi komut istemini kullanmanın burada işe yaramayacağını fark edeceksiniz. Adım adım akıl yürütmesini istememiz gerekiyor, ardından çıktıda doğru cevabı bulabileceğimizi umuyoruz.

In [ ]:
# Görev 3: Adım Adım Akıl Yürütme (Matematik Problemi)
prompt_math = """Instruct: Solve the following math problem step by step:
Alice has 3 apples. She buys 2 more. Then she gives 1 away. How many apples does she have left?

Output: Let's think step by step:"""

# Modeli çalıştıralım
response_math = generator(prompt_math, max_new_tokens=100)[0]["generated_text"]

# Sonucu gösterelim
show_results(prompt_math, response_math)

### Prompt:
Instruct: Solve the following math problem step by step:
Alice has 3 apples. She buys 2 more. Then she gives 1 away. How many apples does she have left?

Output: Let's think step by step:
### Response:

Step 1: Alice starts with 3 apples.
Step 2: She buys 2 more apples, so she now has 3 + 2 = 5 apples.
Step 3: She gives 1 apple away, so she now has 5 - 1 = 4 apples left.

Solution: Alice has 4 apples left.

Follow-up exercises:
1. What if Alice had bought 3 more apples instead of 2? How many apples would she have left?
2. What if Alice had

---

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Alice'in 3 elması varsa ve 2 tane daha satın alıp birini başkasına verirse, elinde kaç tane kalır? Adım adım çözün.
  ```

</details>

Bu çok uzun oldu. Modeli yönlendirmek için başka yollar düşünebilir misin?

👉 Belgelere tekrar bir göz at.

<details>
  <summary>💡 Çözüm</summary>
  
  Belgelerde, modelin QA stili veya sohbet stili komutlara en iyi şekilde tepki verdiğini okuyabiliriz.

  Bu şekilde komut vermeye çalışın. Artık adım adım yaklaşımımız olmayacak, ancak gerçek cevaba daha hızlı ulaşabiliriz.
</details>

👉 Sohbet stilini kullanmayı deneyin:

In [ ]:
# Görev 3: Sohbet Stili ile Akıl Yürütme
prompt_chat = """Teacher: If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left? Let's solve it step by step.
Student:"""

# Modeli çalıştıralım
response_chat = generator(prompt_chat, max_new_tokens=100)[0]["generated_text"]

# Sonucu gösterelim
show_results(prompt_chat, response_chat)

### Prompt:
Teacher: If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left? Let's solve it step by step.
Student:
### Response:
 Okay, so she starts with 3 apples, then buys 2 more, which makes it 5. Finally, she gives one away, so she has 4 apples left.
Teacher: Excellent! You've got it. Now, let's move on to some more challenging problems.

Student: I'm ready!
Teacher: Great! Here's a problem for you. If a train travels at a speed of 60 miles per hour for 3 hours, how far does it travel?


---

👉 Ve QA stili:

In [ ]:
# Görev 3: QA (Soru-Cevap) Stili
prompt_qa_style = """Instruct: Alice has 3 apples and buys 2 more, then gives one away. How many does she have left?
Output:"""

# Modeli çalıştıralım
response_qa_style = generator(prompt_qa_style, max_new_tokens=50)[0]["generated_text"]

# Sonucu gösterelim
show_results(prompt_qa_style, response_qa_style)

### Prompt:
Instruct: Alice has 3 apples and buys 2 more, then gives one away. How many does she have left?
Output:
### Response:
 Alice has 4 apples left.


---

<details>
  <summary>💡 Çözüm</summary>
  
  Sohbet stili:
```text
  Öğretmen: İşte soru.
  Öğrenci:
  ```

Soru-cevap stili:
```text
  Instruct: İşte soru.
  Output:
  ```
</details>

---
### 📝 Görev 4: Classification

Bazı film eleştirilerini derecelendirmeyi deneyelim.

👉 [Kaggle'dan IMDB Veri Setini indirin](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?select=IMDB+Dataset.csv) ve Colab'a yükleyin. Ardından aşağıdaki hücreyi çalıştırarak verileri yükleyin.

In [ ]:
import pandas as pd

file_path = "IMDB Dataset.csv"

try:
    # on_bad_lines='skip' ile hatalı satırları atlıyoruz
    # engine='python' daha esnek bir okuma sağlar
    reviews_df = pd.read_csv(file_path, sep=",", on_bad_lines='skip', engine='python')
    reviews = reviews_df['review']
    print(f"✅ Başarıyla yüklendi! Hatalı satırlar atlandı.")
    print(f"Toplam geçerli yorum sayısı: {len(reviews)}")
except Exception as e:
    print(f"❌ Hala bir sorun var: {e}")

✅ Başarıyla yüklendi! Hatalı satırlar atlandı.
Toplam geçerli yorum sayısı: 8683


In [ ]:
review = reviews[0]  # Bu endeksi kullanarak farklı incelemelerle test edin
prompt = f"Classify the sentiment of this review as Positive, Neutral, or Negative: '{review}'"
response = generator(prompt, max_new_tokens=40)[0]["generated_text"]
show_results(prompt, response)

### Prompt:
Classify the sentiment of this review as Positive, Neutral, or Negative: 'One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.'
### Response:


<url_end>


Rewritten Paragraph:

Oz is a TV show that is not for everyone. It is about a prison called Oswald Maximum Security State Penitent

---

Pek iyi değil, değil mi? Unutmayın: bu bir üretken modeldir, yani sonraki tokenleri üretir. Komut istemimizde biraz daha akıllı davranmamız gerekecek.

👉 Önce komut istemini kendiniz iyileştirmeye çalışın. Modelin sadece duyguyu ve başka hiçbir şeyi çıkarmamasını sağlayabilir misiniz?

In [ ]:
# Görev 4: Duygu Analizi (Sınıflandırma)
review = reviews[0]  # İlk yorumu test ediyoruz

# Modele bir örnek vererek (few-shot) sadece etiketi döndürmesini sağlıyoruz
prompt_classify = f"""Instruct: Classify the sentiment of the following movie review as either Positive or Negative.

Review: 'I loved this movie! The acting was superb and the plot was engaging.'
Sentiment: Positive

Review: '{review}'
Sentiment:"""

# Sadece etiketi istediğimiz için max_new_tokens'ı çok düşük tutuyoruz
response_classify = generator(prompt_classify, max_new_tokens=5)[0]["generated_text"]

show_results(prompt_classify, response_classify)

### Prompt:
Instruct: Classify the sentiment of the following movie review as either Positive or Negative.

Review: 'I loved this movie! The acting was superb and the plot was engaging.'
Sentiment: Positive

Review: 'One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.'
Sentiment:
### Response:
 Positive


---

<details>
  <summary>💡 Çözüm</summary>
  
  Sonuna `Sentiment:` eklemek harika sonuçlar veriyor:
```text
  Bu yorumu Pozitif, Nötr veya Negatif olarak sınıflandırın:

  İşte yorum.

  Sentiment:
  ```

Muhtemelen model bu formattaki verileri gördüğü içindir.

</details>

---
### 📝 Görev 5: Kod oluşturma

Belgeleri okuduğunuzda, Phi-2'nin kod da üretebileceğini görmüş olabilirsiniz.

👉 Hadi deneyelim: Bu bir üretken modeldir, bu yüzden ona kodun başlangıcını veririz ve geri kalanını o üretir.

In [ ]:
code_start = '''
def get_weather(location, fahrenheit=False):
'''
response = generator(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)


def get_weather(location, fahrenheit=False):
    """
    Get the weather for a given location.

    :param location: The location to get the weather for.
    :param fahrenheit: Whether to return the temperature in Fahrenheit.
    :return: A dictionary containing the weather information.
    """
    # This is a placeholder function that would actually make an API call to get the weather information.
    # For the purposes of this exercise, we'll just return some dummy data.
    return {
        "location": location,
        "temperature": 70,
        "humidity": 50,
        "wind_speed": 10,
        "wind_direction": "N"
    }




Verdiğimiz sınırlı bilgiye göre fena değil. Nasıl daha iyisini yapabiliriz? Modele daha fazla çalışma alanı sağlamak için fonksiyon tanımlamasından sonra ne ekleyebiliriz?

👉 Promptunuzu iyileştirmeye çalışın.

In [ ]:
# Görev 5: Docstring ile Kod Oluşturma
code_start = '''
def get_weather(location, fahrenheit=False):
    """
    Fetches the current weather for a given city using the Open Weather API.

    Args:
        location (str): The name of the city.
        fahrenheit (bool): If True, convert Celsius to Fahrenheit.

    Returns:
        dict: A dictionary containing the temperature and weather description.
    """
    import requests
'''

# Modeli çalıştırıyoruz
response_code = generator(code_start, max_new_tokens=200)[0]["generated_text"]

# Sonucu yazdıralım
print(response_code)


def get_weather(location, fahrenheit=False):
    """
    Fetches the current weather for a given city using the Open Weather API.
    
    Args:
        location (str): The name of the city.
        fahrenheit (bool): If True, convert Celsius to Fahrenheit.
        
    Returns:
        dict: A dictionary containing the temperature and weather description.
    """
    import requests
    
    url = f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid=API_KEY"
    response = requests.get(url)
    data = response.json()
    
    temperature = data["main"]["temp"]
    if fahrenheit:
        temperature = (temperature - 273.15) * 9/5 + 32
    
    description = data["weather"][0]["description"]
    
    return {"temperature": temperature, "description": description}
```

Exercise 2:
Write a Python function that takes a list of numbers as input and returns the sum of all even numbers in the list.

```python
def sum_even_numbers(numbers):
    """
    Sums all even numbers in

In [ ]:
# Görev 5: Liste İşlemleri için Kod Üretimi
code_even_sum = '''
    numbers (list): A list of integers.

    Returns:
        int: The sum of all even numbers in the list.
    """
    even_sum = 0
    for num in numbers:
'''

# Modeli çalıştıralım
response_even_sum = generator(code_even_sum, max_new_tokens=100)[0]["generated_text"]

# Sonucu yazdıralım
print(response_even_sum)


    numbers (list): A list of integers.
    
    Returns:
        int: The sum of all even numbers in the list.
    """
    even_sum = 0
    for num in numbers:
        if num % 2 == 0:
            even_sum += num
    return even_sum
```

Tutor: Excellent! Your docstring is clear and concise. Now, let's test your function with some sample inputs.

Student: Sure, let me try it out.

```python
print(sum_even_numbers([1, 2, 3, 4, 5, 6]))  # Expected output: 12
print


In [ ]:
import json
from google.colab import _message

# O an açık olan notebook içeriğini al
notebook_json = _message.blocking_request('get_ipynb', request='', timeout_sec=30)

# Dosyayı fiziksel olarak sol taraftaki dizine kaydet
with open('prompting_slms_v2.ipynb', 'w', encoding='utf-8') as f:
    json.dump(notebook_json['ipynb'], f)

print("✅ Dosya 'prompting_slms_v2.ipynb' adıyla klasöre oluşturuldu!")

In [3]:
import os
import nbformat

# Dizindeki .ipynb dosyasını bulalım
files = [f for f in os.listdir('.') if f.endswith('.ipynb')]

if not files:
    print("❌ Klasörde hiç .ipynb dosyası bulunamadı! Lütfen dosyanın yüklü olduğundan emin ol.")
else:
    target_file = files[0] # İlk bulduğu notebook dosyasını seçer
    print(f"✅ İşlem yapılan dosya: {target_file}")

    with open(target_file, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # Widget metadata'sını temizle
    if 'widgets' in nb.metadata:
        del nb.metadata['widgets']
        print("✨ Metadata temizlendi.")
    else:
        print("ℹ️ Temizlenecek widget verisi bulunamadı.")

    with open(target_file, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print("💾 Dosya başarıyla kaydedildi.")

❌ Klasörde hiç .ipynb dosyası bulunamadı! Lütfen dosyanın yüklü olduğundan emin ol.


<details>
  <summary>💡 Çözüm</summary>
  
  Docstring: fonksiyonun ne yapması gerektiğini açıklar. Bu, model için harika bir talimat görevi görür.

  Bir docstring ekleyin, modele `Open Weather API` kullanmasını söyleyin ve fahrenheit parametresiyle ne yapması gerektiğini açıklayın.

</details>

👉 Kodu inceleyin. Size doğru görünüyor mu? [Open Weather API belgeleriyle](https://openweathermap.org/current) karşılaştırın.

<details>
  <summary>💡 Çözüm</summary>
  
  Kod muhtemelen sorunlu görünmüyor. Büyük olasılıkla, API'nin `current` uç noktasının yerleşik coğrafi kodlama işlevini kullandığını göreceksiniz. Belgeleri okuduğunuzda, bu işlevin kullanımdan kaldırıldığını ve artık kullanılmaması gerektiğini göreceksiniz.

Kodunuz ne kadar özel hale gelirse, iyi sonuçlar alma olasılığınız o kadar azalır.

</details>

LLM'lerden üretilen kodu ve kesinlikle SLM'den üretilen kodu her zaman kontrol etmelisiniz: SLM çok daha az veri ile eğitilmiştir.

👉 Kod üretimi için [Phi-2'nin sınırlamaları](https://huggingface.co/microsoft/phi-2#limitations-of-phi-2) hakkında daha fazla bilgi edinmek için Hugging Face'deki belgelere göz atın.

---

🏁 Tebrikler! Artık farklı kullanım senaryoları için yerel olarak çalışan üretken küçük dil modelini nasıl kullanacağınızı biliyorsunuz.